## Workspace setup

In [1]:
from datetime import datetime  
import uproot
from functools import partial
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import tensorflow_models as tfm
import tensorflow.keras.layers as layers

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

import importlib

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys, os

home = os.getenv("HOME")
if home == "/home/jovyan":
    sys.path.append("/home/jovyan/ELITPC/TPCReco/PythonAnalysis/python/")
else:
    sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")
    os.chdir('/home/akalinow/scratch/ELITPC/PythonAnalysis/')

os.makedirs("fig_png", exist_ok=True)

2025-12-19 08:38:55.438000: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Training dataset preparation

In [3]:
import io_functions as io
importlib.reload(io)

import plotting_functions as plf
importlib.reload(plf)

import utility_functions as utils
importlib.reload(utils)

batchSize = 64
dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_gun_MC_200k_filtered_length_30-100mm', compression="GZIP")
dataset = dataset.batch(batchSize, drop_remainder=True)
dataset = dataset.map(lambda x: x['sim'])
dataset = dataset.map(lambda x,y: (tf.reshape(x, (-1,)+io.projections.shape), tf.reshape(y, (-1,3,3))))
dataset = dataset.map(lambda x,y: (x, utils.XYZtoUVWT_event(y)))
dataset = dataset.map(lambda x,y: (x, tf.reshape(y, (-1,12))))
#dataset = dataset.take(1000).cache('/scratch_ssd/akalinow/data_cache/MergedEvent_Track3D_TwoProng_gun_MC_200k_filtered_length_30-100mm_UVWT_cache').prefetch(tf.data.AUTOTUNE)
#tfds.benchmark(dataset)

input_spec = tf.keras.layers.InputSpec(shape=(None,)+next(iter(dataset))[0].shape[1:])
output_spec = tf.keras.layers.InputSpec(shape=next(iter(dataset))[1].shape[1:])

## Model definition

In [ ]:
#######################################################
class timeDeltaLayer(tf.keras.Layer):
    def __init__(self, num_outputs):
      super(timeDeltaLayer, self).__init__()
      self.num_outputs = num_outputs

    def call(self, inputs):
      outputs = inputs +  inputs[:,2:3]*tf.constant([[0,0,0,0, 0,0,0,1, 0,0,0,1]], dtype=float)
      return outputs
########################################################
########################################################
class StraightTrackLoss(tf.keras.Loss):

    def __init__(self):
      super(StraightTrackLoss, self).__init__()

    def call(self, y_true, y_pred):
        delta_U = y_pred[:,4::4] - y_pred[:,0:1]
        delta_V = y_pred[:,5::4] - y_pred[:,1:2]
        delta_W = y_pred[:,6::4] - y_pred[:,2:3]
        delta_T = (y_pred[:,7::4] - y_pred[:,3:4])
        weights = tf.stack((delta_T[:,1], -delta_T[:,0]), axis=1)/(tf.math.reduce_prod(delta_T, keepdims=True, axis=1) + 0.001)

        u_term = tf.math.reduce_sum(delta_U*weights, axis=1)
        v_term = tf.math.reduce_sum(delta_V*weights, axis=1)
        w_term = tf.math.reduce_sum(delta_W*weights, axis=1)
        loss = tf.math.reduce_mean(u_term**2 + v_term**2 + w_term**2)
        loss += tf.keras.losses.MeanSquaredError(y_true, y_pred)
        return loss
########################################################
########################################################
def getModel():

  model = tf.keras.Sequential([
  tf.keras.layers.Input(shape=input_spec.shape[1:], name="input_image", dtype=tf.float32),
  tf.keras.layers.Resizing(height=256, width=256), 
  tf.keras.layers.Conv2D(16, 4, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(32, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(64, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(output_spec.shape[0], activation="linear"),
  #timeDeltaLayer(output_spec.shape[0])
  ])

  initial_learning_rate = 0.01
  lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=836,
                  decay_rate=0.98,
                  staircase=False)

  optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
  model.compile(optimizer = optimizer, 
                #loss = 'mse', 
                loss = StraightTrackLoss(),
                metrics=['mse', 'mape']) 

  return model

In [ ]:
backbone = tfm.vision.backbones.VisionTransformer(
      mlp_dim=1,
      num_heads=1,
      num_layers=1,
      attention_dropout_rate=0.0,
      dropout_rate=0.0,
      init_stochastic_depth_rate=0.0,
      input_specs=input_spec,
      patch_size=16,
      hidden_size=64,
      representation_size=12,
      pooler='token',
      kernel_regularizer=None,
      output_2d_feature_maps=True
  )
backbone.summary()


x = np.random.rand(1, input_spec.shape[1], input_spec.shape[2], input_spec.shape[3]).astype('float32')
y = backbone(x)
print(y.keys())
print(y['pre_logits'].shape)
print(backbone.output_specs)

In [ ]:
class VisionTransformerModel(tf.keras.Model):

  def __init__(self, vit, head):
    super(VisionTransformerModel, self).__init__()
    self.vit = vit
    self.head = head

  def call(self, inputs):
    x = self.vit(inputs)
    key = list(self.vit.output_specs.keys())[0]
    x = self.head(x[key])
    return x
  
  def summary(self,
              line_length=None,
              positions=None,
              print_fn=None,
              expand_nested=False,
              show_trainable=False,
              layer_range=None):
    self.vit.summary(line_length=line_length,
                     positions=positions,
                     print_fn=print_fn,
                     expand_nested=expand_nested,
                     show_trainable=show_trainable,
                     layer_range=layer_range)
    self.head.summary(line_length=line_length,
                      positions=positions,
                      print_fn=print_fn,
                      expand_nested=expand_nested,
                      show_trainable=show_trainable,
                      layer_range=layer_range)


def getVITModel():

  backbone = tfm.vision.backbones.VisionTransformer(
      mlp_dim=64,
      num_heads=4,
      num_layers=4,
      attention_dropout_rate=0.0,
      dropout_rate=0.0,
      init_stochastic_depth_rate=0.0,
      input_specs=input_spec,
      patch_size=8,
      hidden_size=8,
      representation_size=output_spec.shape[0],
      pooler='token',
      kernel_regularizer=None,
      output_2d_feature_maps=True
  )
  feature_maps_shape = next(iter(backbone.output_specs.values()))[1:]

  head_1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=feature_maps_shape, name="vit_output", dtype=tf.float32),
    tf.keras.layers.Conv2D(16, 4, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(32, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(64, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(output_spec.shape[0], activation="linear")
    ])

  head = tf.keras.Sequential([
    tf.keras.layers.Input(shape=backbone.layers[-1].output_shape[1:], name="vit_output", dtype=tf.float32),
    tf.keras.layers.Flatten(),
      tf.keras.layers.Dense(16, activation='relu',
                            kernel_initializer=tf.keras.initializers.HeNormal(),
                            bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                            kernel_initializer=tf.keras.initializers.HeNormal(),
                            bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                            kernel_initializer=tf.keras.initializers.HeNormal(),
                            bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                            kernel_initializer=tf.keras.initializers.HeNormal(),
                            bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                            kernel_initializer=tf.keras.initializers.HeNormal(),
                            bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(output_spec.shape[0], activation="linear")
      ])
  
  model = VisionTransformerModel(vit=backbone, head=head_1)
  
  initial_learning_rate = 0.01
  lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=836,
                  decay_rate=0.98,
                  staircase=False)

  optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
  model.compile(optimizer = optimizer,
                loss = 'mse', 
                metrics=['mse', 'mape']) 
  
  return model

In [ ]:
model = getModel()
model(tf.random.normal((1,)+input_spec.shape[1:]))
model.summary()

## Model training

In [ ]:
%%time

import plotting_functions as plf
importlib.reload(plf)

log_dir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
#tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, profile_batch=(10, 20))
#early_stop_callback = tf.keras.callbacks.EarlyStopping(patience=2, verbose=1)
#callbacks =  [early_stop_callback, tensorboard_callback]

model = getModel()
#model = getVITModel()
model.trainable = True

initial_learning_rate = 0.01
decay_steps = dataset.cardinality().numpy()
if decay_steps == tf.data.INFINITE_CARDINALITY or decay_steps == tf.data.UNKNOWN_CARDINALITY:
    decay_steps = 1000


print("Decay steps:", decay_steps)    

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=decay_steps,
                  decay_rate=0.98,
                  staircase=False)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse']) 


epochs=60
history = model.fit(dataset.skip(10),
                    epochs=epochs,
                    verbose = 1,
                    validation_data = dataset.take(10),
                    #callbacks=callbacks
                    )
plf.plotTrainHistory(history)

current_time = datetime.now().strftime("%Y_%b_%d_%H_%M_%S")
print("Training start. Current Time =", current_time)

job_dir = f"training/{epochs:04d}_"+current_time+".keras"
model.save(job_dir)

job_dir = f"training/{epochs:04d}_"+current_time+"/"
model.export(job_dir)

## Model performance on training data.

Fill Pandas DataFrame with true and response values.

In [4]:
%%time
import utility_functions as utils
importlib.reload(utils)
import pandas as pd

nBatches = 100
data = np.zeros_like(utils.getSimRecoColumns(utils.columnsUVWT).reshape(1,-1)) 


path = '/home/akalinow/scratch/ELITPC/PythonAnalysis/training/0060_2025_Dec_18_22_42_18.keras'
model = tf.keras.models.load_model(path)


for aBatch in dataset.take(nBatches):

    features = aBatch[0]
    labels = aBatch[1].numpy()
    modelAnswer = model(features).numpy()
    data = np.append(data, np.column_stack((labels,modelAnswer)), axis=0)

df = pd.DataFrame(data=data[1:], columns = utils.getSimRecoColumns(utils.columnsUVWT), dtype=np.float32)
df.describe()    

2025-12-19 08:40:22.984408: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8906


CPU times: user 32.3 s, sys: 9.05 s, total: 41.3 s
Wall time: 22.7 s


2025-12-19 08:40:44.467707: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,uVtx_sim,vVtx_sim,wVtx_sim,tVtx_sim,uAlpha_sim,vAlpha_sim,wAlpha_sim,tAlpha_sim,uCarbon_sim,vCarbon_sim,...,wVtx_reco,tVtx_reco,uAlpha_reco,vAlpha_reco,wAlpha_reco,tAlpha_reco,uCarbon_reco,vCarbon_reco,wCarbon_reco,tCarbon_reco
count,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,...,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000
mean,66.498276,112.818558,113.153610,142.123199,66.743820,111.262970,111.352470,140.993073,66.443039,112.523651,...,113.048363,139.170944,66.255989,109.843300,110.126198,141.725372,66.091309,112.555901,112.845451,139.109924
std,0.671799,33.126114,33.132992,99.553474,31.737627,46.354370,46.374043,125.540665,4.707214,33.539555,...,29.455368,99.930496,14.396978,40.305496,40.692600,125.140900,2.230922,28.609829,28.862452,119.417427
min,64.139458,54.581097,55.110489,56.129852,0.666667,-0.000004,-0.166672,5.000000,53.249435,47.813690,...,57.326202,52.707867,7.453279,11.338568,2.057119,3.009140,58.468018,52.760731,53.912636,49.955505
25%,66.042553,84.502508,84.791370,75.042160,45.920155,78.164890,78.620125,56.142773,62.562115,84.041840,...,83.164335,71.794476,60.581820,84.464851,85.543224,56.684387,64.909466,83.715958,84.245722,56.536491
50%,66.494247,113.169170,113.507355,95.190056,66.576904,111.223873,112.245617,61.383003,66.496521,112.979858,...,114.551746,89.483025,65.016663,108.445057,107.769199,62.951124,66.176636,114.271545,114.617504,61.311636
75%,66.966661,141.073429,141.610519,178.370609,87.934475,144.103954,144.154762,201.512691,70.350586,140.625183,...,141.994919,174.452908,72.527037,138.612022,134.796860,210.672539,67.061602,140.458462,140.623142,192.499390
max,68.987480,171.261978,171.717407,445.448364,132.333328,226.000000,226.833328,516.286194,78.497955,179.634109,...,164.251694,449.508545,131.424789,226.948746,254.986938,559.226318,75.786331,163.014343,163.262314,495.878754


### Resolution plots

In [7]:
import plotting_functions as plf
importlib.reload(plf)

#plf.controlPlots(df)
plf.plotEndPointRes(df=df, edge="Vtx", coordinates=["u", "v", "w", "t"])
plf.plotEndPointRes(df=df, edge="Alpha", coordinates=["u", "v", "w", "t"])
plf.plotEndPointRes(df=df, edge="Carbon", coordinates=["u", "v", "w", "t"])

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

plt.show()

<Figure size 800x800 with 4 Axes>

<Figure size 800x800 with 4 Axes>

<Figure size 800x800 with 4 Axes>

<Figure size 800x800 with 4 Axes>

<Figure size 800x800 with 4 Axes>

<Figure size 800x800 with 4 Axes>

<Figure size 800x800 with 4 Axes>

<Figure size 800x800 with 4 Axes>

<Figure size 800x800 with 4 Axes>